# Covid-19 Tracking Dashboard

This dashboard displays data about Covid-19, using data from the UK Health Security Agency (UKHSA)

In [2]:
%pip install ipywidgets pandas numpy matplotlib requests

  Using cached ipywidgets-8.1.9-py3-none-any.whl.metadata (2.4 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached widgetsnbextension-4.0.16-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.17-py3-none-any.whl.metadata (20 kB)
  Using cached tzdata-2026.4-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
Using cached ipywidgets-8.1.9-py3-none-any.whl (140 kB)
Using cached jupyterlab_widgets-3.0.17-py3-none-any.whl (217 kB)
Using cached widgetsnbextension-4.0.16-py3-none-any.whl (2.2 MB)
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ------------------------- -------------- 6.3/9.8 MB 33.5 MB/s eta 0:

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from IPython.display import clear_output
import ipywidgets as wdg
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import time
import json

ModuleNotFoundError: No module named 'ipywidgets'

In [ ]:
%matplotlib inline
# make figures larger
plt.rcParams['figure.dpi'] = 100

In [ ]:
class APIwrapper:
    # class variables shared among all instances
    _access_point="https://api.ukhsa-dashboard.data.gov.uk"
    _last_access=0.0 # time of last api access
    
    def __init__(self, theme, sub_theme, topic, geography_type, geography, metric):
        """ Init the APIwrapper object, constructing the endpoint from the structure
        parameters """
        # build the path with all the required structure parameters. You do not need to edit this line,
        # parameters will be replaced by the actual values when you instantiate an object of the class!
        url_path=(f"/themes/{theme}/sub_themes/{sub_theme}/topics/{topic}/geography_types/" +
                  f"{geography_type}/geographies/{geography}/metrics/{metric}")
        # our starting API endpoint
        self._start_url=APIwrapper._access_point+url_path
        self._filters=None
        self._page_size=-1
        # will contain the number of items
        self.count=None

    def get_page(self, filters={}, page_size=5):
        """ Access the API and download the next page of data. Sets the count
        attribute to the total number of items available for this query. Changing
        filters or page_size will cause get_page to restart from page 1. Rate
        limited to three request per second. The page_size parameter sets the number
        of data points in one response page (maximum 365); use the default value 
        for debugging your structure and filters. """
        # Check page size is within range
        if page_size>365:
            raise ValueError("Max supported page size is 365")
        # restart from first page if page or filters have changed
        if filters!=self._filters or page_size!=self._page_size:
            self._filters=filters.copy()
            self._page_size=page_size
            self._next_url=self._start_url
        # signal the end of data condition
        if self._next_url==None: 
            return [] # we already fetched the last page
        # simple rate limiting to avoid bans
        curr_time=time.time() # Unix time: number of seconds since the Epoch
        deltat=curr_time-APIwrapper._last_access
        if deltat<0.33: # max 3 requests/second
            time.sleep(0.33-deltat)
        APIwrapper._last_access=curr_time
        # build parameter dictionary by removing all the None
        # values from filters and adding page_size
        parameters={x: y for x, y in filters.items() if y!=None}
        parameters['page_size']=page_size
        # the page parameter is already included in _next_url.
        # This is the API access. Response is a dictionary with various keys.
        # the .json() method decodes the response into Python object (dictionaries,
        # lists; 'null' values are translated as None).
        response = requests.get(self._next_url, params=parameters).json()
        # update url so we'll fetch the next page
        self._next_url=response['next']
        self.count=response['count']
        # data are in the nested 'results' list
        return response['results'] 

    def get_all_pages(self, filters={}, page_size=365):
        """ Access the API and download all available data pages of data. Sets the count
        attribute to the total number of items available for this query. API access rate
        limited to three request per second. The page_size parameter sets the number
        of data points in one response page (maximum 365), and controls the trade-off
        between time to load a page and number of pages; the default should work well 
        in most cases. The number of items returned should in any case be equal to 
        the count attribute. """
        data=[] # build up all data here
        while True:
            # use get_page to do the job, including the pacing
            next_page=self.get_page(filters, page_size)
            if next_page==[]:
                break # we are done
            data.extend(next_page)
        return data

In [ ]:
# Loading initial data from disk 
# This code loads "canned" data from my JSON files and assigns it to the jsondata dictionary

jsondata={}
with open("north_west_cases.json", "rt") as INFILE:
    jsondata['north_west'] = json.load(INFILE) # Reads text insde JSON file and stores in dictionary
        
with open("north_east_cases.json", "rt") as INFILE:
    jsondata['north_east'] = json.load(INFILE)
        
with open("london_cases.json", "rt") as INFILE:
    jsondata['london'] = json.load(INFILE)
        
with open("mean_cases_north_west.json", "rt") as INFILE:
    jsondata['mean_cases_north_west'] = json.load(INFILE)
    
with open("mean_cases_north_east.json", "rt") as INFILE:
    jsondata['mean_cases_north_east'] = json.load(INFILE)
    
with open("mean_cases_london.json", "rt") as INFILE:
    jsondata['mean_cases_london'] = json.load(INFILE)

In [ ]:
# Wrangling the data
# This code contains the logic to wrangle the raw data into dataframes that can be used for plotting the data on graphs
# putting the wrangling code into a function allows you to call it again after refreshing the data through 

# Function for first graph
def wrangle_data(rawdata): 
    """ Parameters: rawdata - data from json file or API call. Returns a dataframe.
    Edit to include the code that wrangles the data, creates the dataframe and fills it in. """
    
    def parse_date(datestring):
        """ Convert a date string into a pandas datetime object """
        return pd.to_datetime(datestring, format="%Y-%m-%d")

# Creates list mapping the data sources to a column name 
    datasets_to_process = [
        (rawdata.get('north_west', []), 'north_west_cases'),
        (rawdata.get('north_east', []), 'north_east_cases'),
        (rawdata.get('london', []), 'london_cases')
    ]

# Creates a dictionary to hold data
    data={}
    for dataset, column_name in datasets_to_process: # Loops over datasets_to_process list
        for entry in dataset:
            date=entry['date']
            value=entry['metric_value']
            
            if date not in data:
                data[date]={}
            data[date][column_name]=value

# If the files are empty returns an empty table instead of crashing
    if not data:
        return pd.DataFrame()
    
    dates=list(data.keys())
    dates.sort() # Sorts dates starting from oldest to newest

    startdate = parse_date(dates[0])
    enddate = parse_date(dates[-1])

    index=pd.date_range(startdate, enddate, freq='D') # Creates an empty dataframe
    timeseriesdf=pd.DataFrame(index=index, columns=['north_west_cases', 'north_east_cases', 'london_cases'])

# Fills the dataframe
    for date, entry in data.items():  
        pd_date=parse_date(date) # converts to Pandas format
        for column in ['north_west_cases', 'north_east_cases', 'london_cases']: 
            value= entry.get(column, 0.0)  
            timeseriesdf.loc[pd_date, column]=value
                
    # fill in any remaining "holes" due to missing dates
    timeseriesdf.fillna(0.0, inplace=True)
        
    return timeseriesdf
    
# Function for second graph
def wrangle_mean_data(rawdata):
    """ Parameters: rawdata - dict containing the mean data lists. Returns meandf. """
    def parse_date(datestring):
        return pd.to_datetime(datestring, format="%Y-%m-%d")

    datasets_to_process = [
        (rawdata.get('mean_cases_north_west', []), 'mean_cases_north_west'),
        (rawdata.get('mean_cases_north_east', []), 'mean_cases_north_east'),
        (rawdata.get('mean_cases_london', []), 'mean_cases_london')
    ]

    data = {}
    for dataset, column_name in datasets_to_process:
        for entry in dataset:
            date = entry['date']
            value = float(entry['metric_value']) 
            if date not in data: 
                data[date] = {}
            data[date][column_name] = value

    if not data: return pd.DataFrame()

    dates = list(data.keys())
    dates.sort()
    
    startdate = parse_date(dates[0])
    enddate = parse_date(dates[-1])
    
    index = pd.date_range(startdate, enddate, freq='D')
    meandf = pd.DataFrame(index=index, columns=['mean_cases_north_west', 'mean_cases_north_east', 'mean_cases_london'])
    
    for date, entry in data.items():
        pd_date = parse_date(date)
        for column in ['mean_cases_north_west', 'mean_cases_north_east', 'mean_cases_london']:
            value = entry.get(column, 0.0)
            meandf.loc[pd_date, column] = value
                
    meandf.fillna(0.0, inplace=True)
    
    return meandf

# Save as global variables
timeseriesdf=wrangle_data(jsondata) 
meandf = wrangle_mean_data(jsondata)

In [ ]:
# Download current data

def access_api():
    """ Accesses the UKHSA API. Return data as a like-for-like replacement for the "canned" data loaded from the JSON file. """

# Data for first graph
    structure={
        "theme": "infectious_disease", 
        "sub_theme": "respiratory",
        "topic": "COVID-19",
        "geography_type": "UKHSA%20Region", 
        "geography": "North%20West",
        "metric": "COVID-19_cases_casesByDay"
    }
    api = APIwrapper(**structure)
    north_west_cases = api.get_all_pages()

    structure["geography"] = "North%20East" # Updates value of geography key instead of creating new dictionary
    api = APIwrapper(**structure)
    north_east_cases = api.get_all_pages()

    structure["geography"] = "London"
    api = APIwrapper(**structure)
    london_cases = api.get_all_pages()

# Data for second graph
    structure["metric"] = "COVID-19_cases_rateRollingMean"
    
    structure["geography"] = "North%20West"
    api = APIwrapper(**structure)
    mean_cases_north_west = api.get_all_pages()

    structure["geography"] = "North%20East"
    api = APIwrapper(**structure)
    mean_cases_north_east = api.get_all_pages()

    structure["geography"] = "London"
    api = APIwrapper(**structure)
    mean_cases_london = api.get_all_pages()

# Puts lists into dictionary
    return {
        'north_west': north_west_cases,
        'north_east': north_east_cases,
        'london': london_cases,

        'mean_cases_north_west': mean_cases_north_west,
        'mean_cases_north_east': mean_cases_north_east,
        'mean_cases_london': mean_cases_london
    } # return data read from the API

In [ ]:
# Connect data to refresh button

def api_button_callback(button):
    """ Button callback - it must take the button as its parameter (unused in this case).
    Accesses API, wrangles data, updates global variable df used for plotting. """

# Show loading spinner whilst button is retrieving data
    apibutton.icon = "spinner" 
    apibutton.disabled = True
    apibutton.description = "Retrieving..."

# Wrangles the data and overwrites the dataframes for plotting
    apidata=access_api()
    global timeseriesdf
    global meandf
    
    timeseriesdf=wrangle_data(apidata)
    meandf = wrangle_mean_data(apidata)   
    refresh_graph()
    apibutton.icon="check" # Button changes to check once complete
    apibutton.description = "Success"
    apibutton.disabled = False # Allows the button to be reused
    
apibutton=wdg.Button(
    description='Refresh', 
    disabled=False,
    button_style='info', 
    tooltip="Keep calm and carry on",
    icon='download'
)

# Registers button callback function with the button
apibutton.on_click(api_button_callback) 

display(apibutton)

Button(button_style='info', description='Refresh', icon='download', style=ButtonStyle(), tooltip='Keep calm an…

## Graphs and Analysis

This first graph compares the number of Covid-19 cases across 3 different regions, the North West, North East and London, between 2020 and 2025, displaying the trend in number of infections over time across the country.

You can interact with the graph by using the dropdown box to filter data by region. You can also switch between a linear display of data, and a logarithmic scale, which compresses the data to make it easier to visualize patterns in the data.

Updated data can be obtained by clicking the Refresh button above. This will refresh the data for both graphs.

In [ ]:
# Graphs and analysis

# First graph
def timeseries_graph(gcols, gscale):
    if gscale == 'linear':
        logscale = False
    else:
        logscale = True
    
    if gcols == 'All Regions': # Select all values
        ax = timeseriesdf.plot(
            logy=logscale, 
            figsize=(12, 6)
        )
    else: # Filter by region
        ax = timeseriesdf[[gcols]].plot(
            logy=logscale, 
            figsize=(12, 6)
        )
    
    ax.set_title(f'COVID-19 Cases: {gcols}')
    ax.set_ylabel('Daily Cases')
    ax.set_xlabel('Date')
    ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.show() 

# Second graph
def mean_graph(graphyear):
    if graphyear is None: return
    yeardf = meandf[meandf.index.year == graphyear]
    monthly = yeardf.groupby(pd.Grouper(freq='ME')).mean()
    monthly = monthly.loc[:, (monthly.sum(axis=0) > 0)] #Removes any empty columns in case wrong data loaded previously
    totals = monthly.sum(axis=1)
    monthly = monthly.div(totals, axis=0) * 100
    monthly = monthly[::-1]
    
    ax = monthly.plot(kind='barh', stacked=True, cmap='tab20', figsize=(10, 6))
    ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
    ax.set_yticklabels(monthly.index.strftime('%Y-%m'))
    ax.set_title(f'Share of Case by Region in {graphyear}') # Sets graph title using relevant year
    ax.set_xlabel('Percentage (%)') # Sets title for x axis
    plt.show()

# Widgets
series = wdg.Dropdown(
    options=['All Regions', 'north_west_cases', 'north_east_cases', 'london_cases'],
    value='All Regions',
    description='Regions:',
    disabled=False
)

scale = wdg.RadioButtons(
    options=['linear', 'log'],
    value='linear', # Sets linear as default selection
    description='Scale:',
    disabled=False
)

controls = wdg.HBox([series, scale])

year=wdg.Select(
    options=meandf.index.year.unique(), 
    value=meandf.index.year[-1], 
    rows=1, 
    description='Year',
    disabled=False
)

def refresh_graph():
    """ We change the value of the widget in order to force a redraw of the graph;
    this is useful when the data have been updated. This is a bit of a gimmick; it
    needs to be customised for one of your widgets. """
    current=series.value
    if current==series.options[0]:
        other=series.options[1]
    else:
        other=series.options[0]
    series.value=other 
    series.value=current 


    if len(year.options) > 0:
        current_year = year.value
        if current_year == year.options[0]:
            other_year = year.options[1]
        else:
            other_year = year.options[0]
        year.value = other_year
        year.value = current_year
        
# Connects widgets and graph functions  
graph1 = wdg.interactive_output(timeseries_graph, {'gcols': series, 'gscale': scale})
graph2 = wdg.interactive_output(mean_graph, {'graphyear': year})

# Display first graph
display(controls, graph1)

Output()

This second graph looks at a 7 day rolling average of the number of cases, displaying the data in terms of the proportion that occur across all 3 regions, and grouping the data into monthly averages. This helps to smooth over the data and account for any outlying values that may be found in day to day reports.

This data can be filtered by year using the dropdown list above the graph.

In [ ]:
# Display second graph
display(year, graph2)

Select(description='Year', index=5, options=(2020, 2021, 2022, 2023, 2024, 2025), rows=1, value=2025)

Output()